# Phase 3 Mathematical Model

This notebook rewrites the imported Amari-Chentsov analysis for the KL-projection robotics experiment.
The imported notebook is retained unchanged under `/notebooks/imported/` so positive and negative results remain comparable.


## Introduction

This work is motivated by personalized and embodied AI under strict compute constraints.
The practical goal is not to build a frontier-scale model, but to make a small, locally controlled AI system specialize from its own experience without requiring data-center-scale retraining.
That goal is especially natural for personal AI assistants, AI coworkers, and embodied agents whose useful behavior depends on local context, repeated interaction, and changing preferences.

Existing personalization work argues that prompt-only or retrieval-only customization can be insufficient when users have distinct preferences and changing behavior.
Parameter-efficient personalization methods such as PEFT-U and OPPU therefore store user-specific structure in lightweight trainable parameters rather than relying only on prompt context [1,2].
This notebook shares that motivation, but shifts attention from static user modeling toward continual, embodied adaptation.

Edge and on-device learning work supplies the second motivation.
Autonomous systems cannot always assume fast network access, unlimited power, or centralized training infrastructure.
Continual learning at the edge therefore emphasizes privacy, latency, bandwidth, distribution shift, sustainability, and catastrophic forgetting [3].
Those constraints are not peripheral here; they are the central design pressure.

Robotics adds the third motivation.
Vision-language-action models such as RT-2 show that large pretrained vision-language systems can provide useful semantic priors for robot control [4].
At the same time, real-time deployment pressures motivate smaller or distilled action systems, because large vision-language backbones can be too slow or expensive to run at every control step [5].
The Phase 1/2 system in this project therefore uses a two-speed hierarchy: a slower VLM supplies language, visual reasoning, reward scoring, and strategic check-ins, while a cheaper LSTM handles most action steps.

Phase 3 asks how such a hierarchy can continue learning online.
The implementation scope is deliberately modest: bounded replay, low-rank or diagonal Fisher memory, EWC-style regularization, and reinforcement-learning updates over trainable low-cost components [12,13].
Amari-Chentsov transport remains part of the mathematical background and negative-result record, but it is not an implementation target.
The aim is applied progress: a coherent online-learning mechanism that respects limited compute while preserving enough prior structure to remain useful.


## Related Work

This section is provisional.
The aim is not yet to tell the final story, but to anchor the mathematical model against the nearby literature while the motivation is still being shaped.

Policy-gradient reinforcement learning already rests on the likelihood-ratio, or score-function, identity.
Williams introduced REINFORCE as a statistical gradient-following method for connectionist reinforcement learning [6], and the later policy-gradient theorem formalized the same score-based structure for function approximation [7].
Thus, the score process used below is not new to reinforcement learning; our purpose is to reinterpret it as a frequentist local-estimation object that can also support Fisher-memory mechanics.

Natural policy gradient is the closest established bridge from policy gradients to information geometry.
Kakade's natural policy gradient replaces Euclidean step geometry with Fisher/KL geometry [8], and TRPO later operationalizes the same local-KL idea for large nonlinear policies [9].
These works already show that policy-gradient updates have an information-geometric interpretation.
Our emphasis differs: we are not primarily seeking a natural-gradient optimizer, because inverse-Fisher operations are too expensive for this robotics setting.
Instead, we use the shared score/Fisher structure to justify online memory summaries, EWC-style penalties, replay, and small-step RL updates.

Information geometry and mirror descent provide another neighboring lens.
Amari's natural gradient treats the Fisher information as the Riemannian metric of a statistical model [10], while mirror-descent analyses show how first-order algorithms can induce related non-Euclidean geometries [11].
This literature supports the idea that first-order learning can be interpreted geometrically without always instantiating a full second-order method.

Continual-learning work supplies the memory side of the model.
EWC uses Fisher-based weight regularization to reduce catastrophic forgetting [12], and recent low-rank EWC/LoRA work studies parameter-efficient versions of the same stability-plasticity problem [13].
Those results motivate our Phase 3 scope: keep memory approximate and cheap, make EWC disableable, and treat low-rank or diagonal Fisher summaries as operational tools rather than exact transports.

The imported Amari-Chentsov notebook goes one step further by asking whether Fisher information itself can be transported between nearby parameter values.
That theory is conceptually useful for connecting local frequentist information to moving-target reinforcement learning, but direct Amari-Chentsov transport is not in scope for the current implementation.
Exact likelihood-ratio transport would require retaining old observations or rich sufficient statistics; geometric transport avoids retaining data but asks for high-order curvature, inverse-Fisher, or Hessian-like quantities.
Phase 3 therefore adopts the practical compromise suggested by the literature: bounded replay, low-rank or diagonal Fisher memory, EWC-style regularization, and online RL updates under small parameter movement.



## Bridging Frequentism to REINFORCE

We observe data while the model is locally fixed at a parameter value $\theta_t$.
Frequentist estimation packages these observations into local score and Fisher summaries.
Reinforcement learning then asks us to move to a nearby parameter value $\theta_{t+1} = \theta_t + d\theta_t$.
The bridge is local: $\|d\theta_t\|$ must be small enough that first-order likelihood-ratio and information-geometric approximations are meaningful.

Let $p_\theta(x)$ be a regular statistical model, let $\ell_\theta(x) := \log p_\theta(x)$, and let

$$ s_i(x;\theta) := \partial_i \ell_\theta(x). $$

For a small displacement $u := d\theta$, the likelihood ratio from $\theta$ to $\theta+u$ satisfies

$$
\frac{p_{\theta+u}(x)}{p_\theta(x)}
= \exp(\ell_{\theta+u}(x)-\ell_\theta(x))
= 1 + u^k s_k(x;\theta) + O(\|u\|^2).
$$

Thus, for any statistic $A(x)$ that is fixed while taking this first-order change,

$$
\mathbb E_{\theta+u}[A(X)]
= \mathbb E_\theta\!\left[A(X)\frac{p_{\theta+u}(X)}{p_\theta(X)}\right]
= \mathbb E_\theta[A(X)] + u^k \mathbb E_\theta[A(X)s_k(X;\theta)] + O(\|u\|^2).
$$

This is the infinitesimal likelihood-ratio transport of information from $\theta$ toward $\theta+u$.
It is exact only to first order, and exact use beyond first order would require retaining observations or sufficiently rich statistics to recompute likelihood ratios.

The Fisher metric is

$$ I_{ij}(\theta) := \mathbb E_\theta[s_i s_j]. $$

If $s_i s_j$ is compared as a transported local tensor rather than naively differentiated as a changing coordinate expression, the first-order likelihood-ratio term contributes

$$
u^k\mathbb E_\theta[s_i s_j s_k] = [C:u]_{ij},
\qquad C_{ijk}:=\mathbb E_\theta[s_i s_j s_k].
$$

Under the e-connection, this is the same first-order object in e-affine coordinates [10].
Writing $P^{(e)}_{\theta+u\to\theta}$ for e-parallel transport back to $T_\theta\Theta$,

$$
P^{(e)}_{\theta+u\to\theta} I(\theta+u)
= I(\theta) + \nabla^{(e)}_u I(\theta) + O(\|u\|^2).
$$

For canonical exponential families, or for a Gaussianized auxiliary score process treated in canonical coordinates,

$$
[\nabla^{(e)}_u I(\theta)]_{ij} = C_{ijk}u^k.
$$

So e-parallel transport is the coordinate-aware, information-geometric form of first-order likelihood-ratio transport.
Outside canonical coordinates, additional Hessian-like terms appear in the ordinary derivative

$$
\partial_k I_{ij}
= \mathbb E_\theta[(\partial_k s_i)s_j + s_i(\partial_k s_j) + s_i s_j s_k],
$$

which is one reason direct Amari-Chentsov transport is not our Phase 3 implementation target.
The bridge remains conceptually useful: local statistical information collected at $\theta_t$ can be transported toward nearby $\theta_{t+1}$ to first order, but exact global transport is computationally expensive.

Now let $\tau$ be a trajectory and let $P_\theta(\tau)$ be the trajectory distribution induced by policy $\pi_\theta$.
Assume environment dynamics do not depend on $\theta$.
For return $R(\tau)$, define

$$ J(\theta) := \mathbb E_{\tau\sim P_\theta}[R(\tau)]. $$

The score-function identity gives

$$
\nabla_\theta J(\theta)
= \nabla_\theta \int R(\tau)P_\theta(\tau)d\tau
= \int R(\tau)P_\theta(\tau)\nabla_\theta\log P_\theta(\tau)d\tau
= \mathbb E_\theta[R(\tau)S_\theta(\tau)],
$$

where

$$ S_\theta(\tau) := \nabla_\theta\log P_\theta(\tau). $$

Because only the policy depends on $\theta$,

$$
S_\theta(\tau) = \sum_{h} \nabla_\theta \log \pi_\theta(a_h\mid s_h),
$$

and therefore

$$
\nabla_\theta J(\theta)
= \mathbb E_\theta\!\left[R(\tau)\sum_h \nabla_\theta\log\pi_\theta(a_h\mid s_h)\right].
$$

This is the REINFORCE gradient [6,7].
The frequentist content is that, at each fixed $\theta_t$, sampled trajectories provide ordinary score observations $S_{\theta_t}(\tau)$ and local information summaries such as

$$
I(\theta_t) = \mathbb E_{\theta_t}[S_{\theta_t}S_{\theta_t}^T].
$$

The same score process therefore supports both frequentist local estimation and policy-gradient learning.
When $\theta_{t+1}=\theta_t+d\theta_t$ is nearby, first-order likelihood-ratio/e-connection transport explains how information collected at $\theta_t$ may be treated as locally informative about $\theta_{t+1}$.
This supplies an axiomatic context for using REINFORCE in a moving-target estimation problem: sample frequentist information locally, summarize it with score/Fisher objects, and update the policy using the reward-weighted score.

The limitation is also part of the model.
Exact transport of all prior information would require retained observations, exact likelihood ratios, or high-order geometry.
Phase 3 therefore targets controlled approximation: bounded replay, low-rank or diagonal Fisher memory, EWC-style regularization, and online RL updates.


## References

[1] Clarke, C., Heng, Y., Tang, L., & Mars, J. (2024). [PEFT-U: Parameter-Efficient Fine-Tuning for User Personalization](https://arxiv.org/abs/2407.18078).

[2] Tan, Z., Zeng, Q., Tian, Y., Liu, Z., Yin, B., & Jiang, M. (2024). [Democratizing Large Language Models via Personalized Parameter-Efficient Fine-tuning](https://arxiv.org/abs/2402.04401).

[3] Pellegrini, L., Lomonaco, V., Graffieti, G., & Maltoni, D. (2021). [Continual Learning at the Edge: Real-Time Training on Smartphone Devices](https://arxiv.org/abs/2105.13127).

[4] Brohan, A. et al. (2023). [RT-2: Vision-Language-Action Models Transfer Web Knowledge to Robotic Control](https://arxiv.org/abs/2307.15818).

[5] Huang, X., Hua, Z., Han, Z., Sural, S., & Rajkumar, R. (2026). [RT-VLA: Real-Time Vision-Language-Action Models via Knowledge Distillation](https://arxiv.org/abs/2606.14010).

[6] Williams, R. J. (1992). [Simple statistical gradient-following algorithms for connectionist reinforcement learning](https://doi.org/10.1007/BF00992696).

[7] Sutton, R. S., McAllester, D., Singh, S., & Mansour, Y. (2000). [Policy Gradient Methods for Reinforcement Learning with Function Approximation](https://proceedings.neurips.cc/paper_files/paper/1999/hash/464d828b85b0bed98e80ade0a5c43b0f-Abstract.html).

[8] Kakade, S. M. (2001). [A Natural Policy Gradient](https://proceedings.neurips.cc/paper_files/paper/2001/hash/4b86abe48d358ecf194c56c69108433e-Abstract.html).

[9] Schulman, J., Levine, S., Moritz, P., Jordan, M. I., & Abbeel, P. (2015). [Trust Region Policy Optimization](https://arxiv.org/abs/1502.05477).

[10] Amari, S. (1998). [Natural gradient works efficiently in learning](https://doi.org/10.1162/089976698300017746).

[11] Raskutti, G., & Mukherjee, S. (2015). [The Information Geometry of Mirror Descent](https://arxiv.org/abs/1310.7780).

[12] Kirkpatrick, J. et al. (2017). [Overcoming catastrophic forgetting in neural networks](https://arxiv.org/abs/1612.00796).

[13] Zheng, Y. et al. (2026). [Revisiting Weight Regularization for Low-Rank Continual Learning](https://arxiv.org/abs/2602.17559).